# Etiquetas a partir de las máscaras `.mrc` (dataset 10490)

Sustituye a `align_tif_to_mrc.py` + las celdas de carga del notebook original.
Hace lo mismo que hacía ese pipeline, pero leyendo las anotaciones `.mrc`:

1. Comprueba que cada máscara tiene el mismo shape que el tomograma (si no, la salta y lo apunta).
2. Fusiona las ~100 máscaras en **un volumen de etiquetas enteras** (`1..N`, 0 = fondo) + `class_map.csv`.
3. Normaliza el tomograma a uint8 (percentiles) y calcula los cortes con anotación (`valid_indices`).
4. Baja las etiquetas a la **rejilla de tokens** de V-JEPA (24x24 por corte) para entrenar después.

Pensado para dejarlo corriendo solo: guarda todo en `output/` y escribe un log (`output/build_labels.log`).
Ejecutar con "Run All". Los ficheros de rutas son relativos a la carpeta del notebook.

In [1]:
import sys; print(sys.executable)

c:\Users\pauum\Desktop\PAULA\uni\MASTER\TFM\CRYOEM\Proyecto\.venv\Scripts\python.exe


In [2]:
from pathlib import Path
from datetime import datetime
import csv, json, time, traceback

import cv2
import mrcfile
import numpy as np
import tifffile
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ----------------------------- CONFIG -----------------------------
BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "datasets_publicos" / "10490"
TOMO_PATH = DATA_DIR / "sample_1_10490.mrc"
ANN_DIR = DATA_DIR / "annotations"

MASK_PATTERN = "*segmentationmask.mrc"
MAX_MASKS = None 

OUT_DIR = BASE_DIR / "output"
PATCH_GRID = 24             # rejilla de tokens de V-JEPA (384/16)
MIN_TOKEN_FRACTION = 0.25   # un token es de la clase X si X cubre >= 25% del parche
PCT_LOW, PCT_HIGH = 1, 99   # percentiles para normalizar el tomograma
# ------------------------------------------------------------------

OUT_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = OUT_DIR / "build_labels.log"
_log_f = open(LOG_PATH, "a", encoding="utf-8")

def log(msg):
    line = f"[{datetime.now():%H:%M:%S}] {msg}"
    print(line, flush=True)
    _log_f.write(line + "\n"); _log_f.flush()

log(f"cwd={BASE_DIR}")
for p in (TOMO_PATH, ANN_DIR):
    if not p.exists():
        raise FileNotFoundError(f"No existe: {p}  (¿estás ejecutando el notebook desde CRYOEM/Proyecto?)")

[00:47:15] cwd=c:\Users\pauum\Desktop\PAULA\uni\MASTER\TFM\CRYOEM\Proyecto


In [3]:
# ---- 1) listar máscaras y leer shapes SOLO de la cabecera ----
def header_shape(path):
    with mrcfile.open(path, permissive=True, header_only=True) as m:
        h = m.header
        return (int(h.nz), int(h.ny), int(h.nx))

def class_name_from(path):
    stem = Path(path).stem
    for suffix in ("_instancesegmentationmask", "_segmentationmask"):
        if stem.endswith(suffix):
            stem = stem[: -len(suffix)]
    return stem.rsplit("-", 1)[0] if "-" in stem else stem   # quita el "-1.0"

tomo_shape = header_shape(TOMO_PATH)
log(f"Tomograma {TOMO_PATH.name}: shape (z,y,x) = {tomo_shape}")

mask_paths = sorted(ANN_DIR.glob(MASK_PATTERN))
if MAX_MASKS:
    mask_paths = mask_paths[:MAX_MASKS]
log(f"{len(mask_paths)} máscaras con patrón {MASK_PATTERN!r}")
if not mask_paths:
    raise FileNotFoundError(f"Ninguna máscara con {MASK_PATTERN} en {ANN_DIR}")

entries, skipped = [], []
for p in mask_paths:
    try:
        shp = header_shape(p)
    except Exception as e:
        skipped.append({"file": p.name, "reason": f"cabecera ilegible: {e}"}); continue
    if shp != tomo_shape:
        skipped.append({"file": p.name, "reason": f"shape {shp} != tomograma {tomo_shape}"}); continue
    entries.append({"path": p, "name": class_name_from(p), "kind": "instance" if "_instance" in p.name else "semantic"})

log(f"Compatibles con el tomograma: {len(entries)} | saltadas: {len(skipped)}")
for s in skipped[:10]:
    log(f"  SALTADA {s['file']}: {s['reason']}")
if not entries:
    raise RuntimeError(
        "Ninguna máscara tiene el mismo shape que el tomograma. "
        "Revisa el log: probablemente hace falta reescalar/alinear (otro binning o recortes)."
    )

[00:47:24] Tomograma sample_1_10490.mrc: shape (z,y,x) = (133, 600, 600)
[00:47:24] 102 máscaras con patrón '*segmentationmask.mrc'
[00:47:25] Compatibles con el tomograma: 102 | saltadas: 0


In [4]:
# ---- 2) primera pasada: vóxeles por clase (para decidir el orden de pintado) ----
def read_mask(path):
    with mrcfile.open(path, permissive=True) as m:
        if m.data is None:
            raise ValueError("sin datos")
        return np.asarray(m.data) > 0          # booleano: >0 = pertenece a la clase

t0 = time.time()
ok = []
for i, e in enumerate(entries, 1):
    try:
        e["voxels"] = int(np.count_nonzero(read_mask(e["path"])))
        ok.append(e)
    except Exception as ex:
        skipped.append({"file": e["path"].name, "reason": f"lectura: {ex}"})
        log(f"  ERROR leyendo {e['path'].name}: {ex}")
    if i % 10 == 0 or i == len(entries):
        log(f"  conteo {i}/{len(entries)} ({time.time()-t0:.0f}s)")
entries = [e for e in ok if e["voxels"] > 0]
log(f"Máscaras con al menos 1 vóxel: {len(entries)}")

# Clases grandes primero, pequeñas después -> las partículas pequeñas no quedan tapadas
entries.sort(key=lambda e: -e["voxels"])
for cid, e in enumerate(entries, 1):
    e["id"] = cid
label_dtype = np.uint8 if len(entries) <= 255 else np.uint16

[00:47:34]   conteo 10/102 (2s)
[00:47:36]   conteo 20/102 (3s)
[00:47:38]   conteo 30/102 (5s)
[00:47:40]   conteo 40/102 (8s)
[00:47:43]   conteo 50/102 (11s)
[00:47:46]   conteo 60/102 (13s)
[00:47:48]   ERROR leyendo peptide_with_high_affinity_for_b_cell_lymphoma2-1.0_instancesegmentationmask.mrc: sin datos
[00:47:48]   conteo 70/102 (15s)


c:\Users\pauum\Desktop\PAULA\uni\MASTER\TFM\CRYOEM\Proyecto\.venv\Lib\site-packages\mrcfile\mrcinterpreter.py:324: RuntimeWarning: Expected 95760000 bytes in data block but limit is 49282048
  warnings.warn(msg, RuntimeWarning)


[00:47:50]   conteo 80/102 (18s)


KeyboardInterrupt: 

In [ ]:
# ---- 3) segunda pasada: pintar el volumen de etiquetas ----
labels = np.zeros(tomo_shape, dtype=label_dtype)
overlap_vox = 0
t0 = time.time()
for i, e in enumerate(entries, 1):
    try:
        m = read_mask(e["path"])
        overlap_vox += int(np.count_nonzero(m & (labels > 0)))
        labels[m] = e["id"]
    except Exception as ex:
        log(f"  ERROR pintando {e['path'].name}: {ex}")
        e["error"] = str(ex)
    if i % 10 == 0 or i == len(entries):
        log(f"  pintado {i}/{len(entries)} ({time.time()-t0:.0f}s)")

fg = int(np.count_nonzero(labels))
log(f"Etiquetas: dtype={labels.dtype}, vóxeles anotados={fg} ({100*fg/labels.size:.2f}%), solapes={overlap_vox}")

tifffile.imwrite(OUT_DIR / "labels_10490.tif", labels, compression="zlib")
np.save(OUT_DIR / "labels_10490.npy", labels)

with open(OUT_DIR / "class_map.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["id", "name", "kind", "voxels", "file"])
    for e in entries:
        w.writerow([e["id"], e["name"], e["kind"], e["voxels"], e["path"].name])
with open(OUT_DIR / "skipped_masks.json", "w", encoding="utf-8") as f:
    json.dump(skipped, f, indent=2, ensure_ascii=False)
log("Guardados: labels_10490.tif/.npy, class_map.csv, skipped_masks.json")

: 

In [ ]:
# ---- 4) tomograma normalizado a uint8 + cortes con anotación ----
with mrcfile.open(TOMO_PATH, permissive=True) as m:
    tomo = np.asarray(m.data, dtype=np.float32)

lo, hi = np.percentile(tomo[::2, ::4, ::4], [PCT_LOW, PCT_HIGH])
tomo_u8 = (np.clip((tomo - lo) / max(hi - lo, 1e-6), 0, 1) * 255).astype(np.uint8)
del tomo

valid_indices = np.where(labels.reshape(labels.shape[0], -1).any(axis=1))[0]
log(f"Cortes con anotación: {len(valid_indices)}/{labels.shape[0]}")

np.save(OUT_DIR / "tomogram_uint8.npy", tomo_u8)
np.save(OUT_DIR / "valid_indices.npy", valid_indices)

: 

In [ ]:
# ---- 5) etiquetas a la rejilla de tokens (por corte): (Z, 24, 24) ----
def slice_to_token_labels(lab2d, grid, min_frac):
    out = np.zeros((grid, grid), dtype=np.int32)
    best = np.zeros((grid, grid), dtype=np.float32)
    for cid in np.unique(lab2d):
        if cid == 0:
            continue
        frac = cv2.resize((lab2d == cid).astype(np.float32), (grid, grid), interpolation=cv2.INTER_AREA)
        upd = (frac >= min_frac) & (frac > best)
        out[upd] = cid
        best[upd] = frac[upd]
    return out

token_labels = np.zeros((labels.shape[0], PATCH_GRID, PATCH_GRID), dtype=np.int32)
valid_set = set(valid_indices.tolist())
t0 = time.time()
for z in range(labels.shape[0]):
    if z in valid_set:
        token_labels[z] = slice_to_token_labels(labels[z], PATCH_GRID, MIN_TOKEN_FRACTION)
    if (z + 1) % 50 == 0:
        log(f"  token labels {z+1}/{labels.shape[0]} ({time.time()-t0:.0f}s)")

np.save(OUT_DIR / "token_labels.npy", token_labels)
n_fg_tokens = int((token_labels > 0).sum())
log(f"token_labels {token_labels.shape}: tokens con clase = {n_fg_tokens} ({100*n_fg_tokens/token_labels.size:.2f}%)")

: 

In [ ]:
# ---- 6) comprobación visual: tomograma + etiquetas superpuestas (guarda PNG) ----
if len(valid_indices):
    picks = valid_indices[np.linspace(0, len(valid_indices) - 1, 4).astype(int)]
    rng = np.random.default_rng(0)
    palette = rng.uniform(0.2, 1.0, size=(int(labels.max()) + 1, 3)); palette[0] = 0
    fig, axs = plt.subplots(2, 4, figsize=(16, 8))
    for k, z in enumerate(picks):
        axs[0, k].imshow(tomo_u8[z], cmap="gray"); axs[0, k].set_title(f"tomograma z={z}"); axs[0, k].axis("off")
        over = np.stack([tomo_u8[z] / 255.0] * 3, axis=-1)
        m = labels[z] > 0
        over[m] = 0.4 * over[m] + 0.6 * palette[labels[z][m]]
        axs[1, k].imshow(over); axs[1, k].set_title("etiquetas"); axs[1, k].axis("off")
    plt.tight_layout()
    plt.savefig(OUT_DIR / "check_labels_overlay.png", dpi=110)
    plt.close()
    log("Guardado output/check_labels_overlay.png  (MÍRALO: las etiquetas deben caer sobre las partículas)")

log("TERMINADO")
_log_f.close()

: 

In [ ]:
# # ---- napari: tomograma + anotaciones ----
# import napari
# OUT = Path("output")
# tomo = np.load(OUT / "tomogram_uint8.npy")
# labels = np.load(OUT / "labels_10490.npy")
# VOXEL = 6.8   # Å, del header del .mrc

# viewer = napari.Viewer(ndisplay=2)
# viewer.add_image(tomo, name="tomograma", scale=(VOXEL,) * 3, contrast_limits=(0, 255))
# viewer.add_labels(labels, name="anotaciones", scale=(VOXEL,) * 3, opacity=0.6)

: 

: 

: 